# U-Net phân đoạn vùng tổn thương trên BUSI

Notebook này là baseline để kiểm tra toàn bộ pipeline trên Kaggle.

> Lưu ý: chia ngẫu nhiên theo ảnh chỉ dùng để thử nghiệm ban đầu. Khi viết bài báo cần chia theo bệnh nhân hoặc dùng fold chính thức của BUS-BRA.


In [ ]:
import os, random, math, json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Tìm đường dẫn dataset

Trước khi chạy, chọn **Add Input** và thêm dataset:

`Breast Ultrasound Images Dataset`


In [ ]:
input_root = Path("/kaggle/input")
candidates = list(input_root.rglob("Dataset_BUSI_with_GT"))

if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy Dataset_BUSI_with_GT. Hãy Add Input dataset BUSI vào notebook."
    )

DATA_DIR = candidates[0]
print("DATA_DIR =", DATA_DIR)


In [ ]:
from dataclasses import dataclass

VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

@dataclass(frozen=True)
class Sample:
    image_path: Path
    mask_paths: tuple

def discover_samples(data_dir, include_normal=False):
    allowed = {"benign", "malignant"}
    if include_normal:
        allowed.add("normal")

    samples = []
    for folder in sorted(Path(data_dir).rglob("*")):
        if not folder.is_dir() or folder.name.lower() not in allowed:
            continue

        images = sorted(
            p for p in folder.iterdir()
            if p.is_file()
            and p.suffix.lower() in VALID_EXTENSIONS
            and "_mask" not in p.stem.lower()
        )

        for image_path in images:
            prefix = image_path.stem + "_mask"
            masks = tuple(sorted(
                p for p in folder.iterdir()
                if p.is_file()
                and p.suffix.lower() in VALID_EXTENSIONS
                and p.stem.startswith(prefix)
            ))

            if masks or folder.name.lower() == "normal":
                samples.append(Sample(image_path, masks))

    return samples

def load_merged_mask(mask_paths, image_size):
    if not mask_paths:
        return Image.new("L", image_size, color=0)

    merged = np.zeros((image_size[1], image_size[0]), dtype=np.uint8)
    for path in mask_paths:
        m = Image.open(path).convert("L")
        if m.size != image_size:
            m = m.resize(image_size, Image.Resampling.NEAREST)
        merged = np.maximum(merged, (np.asarray(m) > 127).astype(np.uint8) * 255)
    return Image.fromarray(merged)

class BUSIDataset(Dataset):
    def __init__(self, samples, image_size=256, augment=False):
        self.samples = samples
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample.image_path).convert("L")
        mask = load_merged_mask(sample.mask_paths, image.size)

        image = TF.resize(
            image, [self.image_size, self.image_size],
            interpolation=InterpolationMode.BILINEAR
        )
        mask = TF.resize(
            mask, [self.image_size, self.image_size],
            interpolation=InterpolationMode.NEAREST
        )

        if self.augment:
            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

            angle = random.uniform(-12, 12)
            image = TF.rotate(
                image, angle,
                interpolation=InterpolationMode.BILINEAR,
                fill=0
            )
            mask = TF.rotate(
                mask, angle,
                interpolation=InterpolationMode.NEAREST,
                fill=0
            )

        return TF.to_tensor(image), (TF.to_tensor(mask) > 0.5).float()

samples = discover_samples(DATA_DIR)
print("Số ảnh có tổn thương:", len(samples))
print("Ví dụ:", samples[0])


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_c, out_c))

    def forward(self, x):
        return self.block(x)

class Up(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = DoubleConv(in_c + skip_c, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)
        x = F.pad(x, [
            diff_x // 2, diff_x - diff_x // 2,
            diff_y // 2, diff_y - diff_y // 2
        ])
        return self.conv(torch.cat([skip, x], dim=1))

class UNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        c = base
        self.inc = DoubleConv(1, c)
        self.down1 = Down(c, c*2)
        self.down2 = Down(c*2, c*4)
        self.down3 = Down(c*4, c*8)
        self.down4 = Down(c*8, c*16)
        self.up1 = Up(c*16, c*8, c*8)
        self.up2 = Up(c*8, c*4, c*4)
        self.up3 = Up(c*4, c*2, c*2)
        self.up4 = Up(c*2, c, c)
        self.outc = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

model = UNet(base=16).to(DEVICE)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
x = torch.randn(2, 1, 256, 256, device=DEVICE)
print("Output:", model(x).shape)


In [ ]:
def dice_loss(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, probs.ndim))
    inter = (probs * targets).sum(dim=dims)
    denom = probs.sum(dim=dims) + targets.sum(dim=dims)
    return 1 - ((2*inter + eps) / (denom + eps)).mean()

bce = nn.BCEWithLogitsLoss()

def criterion(logits, targets):
    return 0.5*bce(logits, targets) + 0.5*dice_loss(logits, targets)

@torch.no_grad()
def dice_metric(logits, targets, eps=1e-7):
    pred = (torch.sigmoid(logits) >= 0.5).float()
    dims = tuple(range(1, pred.ndim))
    inter = (pred * targets).sum(dim=dims)
    denom = pred.sum(dim=dims) + targets.sum(dim=dims)
    return ((2*inter + eps) / (denom + eps)).mean()


## Chia dữ liệu và huấn luyện

Cấu hình dưới đây vừa sức với GPU Kaggle. Có thể tăng `EPOCHS` lên 30–50 sau khi pipeline chạy ổn.


In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 20
VAL_RATIO = 0.2

indices = list(range(len(samples)))
random.Random(SEED).shuffle(indices)
n_val = max(1, int(len(indices) * VAL_RATIO))
val_ids = set(indices[:n_val])

train_samples = [s for i, s in enumerate(samples) if i not in val_ids]
val_samples = [s for i, s in enumerate(samples) if i in val_ids]

train_ds = BUSIDataset(train_samples, IMAGE_SIZE, augment=True)
val_ds = BUSIDataset(val_samples, IMAGE_SIZE, augment=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print("Train:", len(train_ds), "| Val:", len(val_ds))


In [ ]:
model = UNet(base=16).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == "cuda")

best_dice = -1
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    train_dice = 0
    n_train = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        bs = images.size(0)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16 if DEVICE.type == "cuda" else torch.bfloat16,
            enabled=DEVICE.type == "cuda"
        ):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * bs
        train_dice += dice_metric(logits.detach(), masks).item() * bs
        n_train += bs

    model.eval()
    val_loss = 0
    val_dice = 0
    n_val_items = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            bs = images.size(0)

            logits = model(images)
            loss = criterion(logits, masks)

            val_loss += loss.item() * bs
            val_dice += dice_metric(logits, masks).item() * bs
            n_val_items += bs

    row = {
        "epoch": epoch,
        "train_loss": train_loss / n_train,
        "train_dice": train_dice / n_train,
        "val_loss": val_loss / n_val_items,
        "val_dice": val_dice / n_val_items,
    }
    history.append(row)
    print(row)

    if row["val_dice"] > best_dice:
        best_dice = row["val_dice"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "base_channels": 16,
            "image_size": IMAGE_SIZE,
            "best_val_dice": best_dice,
        }, "/kaggle/working/best_unet_busi.pt")
        print("Saved best model.")

print("Best validation Dice:", best_dice)


In [ ]:
# Trực quan một số kết quả
checkpoint = torch.load(
    "/kaggle/working/best_unet_busi.pt",
    map_location=DEVICE
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

images, masks = next(iter(val_loader))
images = images.to(DEVICE)

with torch.no_grad():
    predictions = (torch.sigmoid(model(images)) >= 0.5).float().cpu()

images = images.cpu()
n_show = min(4, len(images))

for i in range(n_show):
    plt.figure(figsize=(12, 3))

    plt.subplot(1, 3, 1)
    plt.imshow(images[i, 0], cmap="gray")
    plt.title("Ảnh")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(masks[i, 0], cmap="gray")
    plt.title("Ground truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(predictions[i, 0], cmap="gray")
    plt.title("U-Net prediction")
    plt.axis("off")

    plt.show()
